# tool_eval — 큰 혼합 네임스페이스에서의 툴콜링 (20개: 관련5 + 방해15)

**세팅:** 도구 20개 중 **5개**는 과제와 관련된 응집된 워크플로(고객지원/주문운영), 나머지 **15개**는
**완전히 다른 컨텍스트**의 방해(distractor) 도구다. 방해 15개는 하위 서브에이전트 3개가 병렬로
실제 도메인별로 촘촘히 작성했다(날씨·항공·환율·DNS·번역 / 영양·음악·토양IoT·천문·세금 / 요리·git·법률·단위·운동).

**측정:** 큰 네임스페이스가 툴콜링에 주는 영향 — A/B 로 비교한다.

| arm | 올라가는 도구 | 재는 것 |
|---|---|---|
| **full(20)** | 관련5 + 방해15 | 매 요청 스키마 비용(입력토큰), 엉뚱한 도구 호출(wrong_tool_calls), 정확도 |
| **clean(5)** | 관련5만 | 기준선 |

관련 도메인은 결정적 데이터셋(고객·주문·청구)이라 정답을 정확 채점한다.
이건 클로드코드의 **ToolSearch/deferred tools** 가 왜 필요한지와 직결된다.

## 0. 셋업

In [ ]:
import os, sys, json, time, re, pathlib
from dataclasses import dataclass, field
from typing import Callable

try:
    from dotenv import load_dotenv
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / ".env").exists():
            load_dotenv(base / ".env"); break
except Exception:
    pass
from openai import OpenAI
MODEL = "gpt-5-nano"
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("OpenAI:", "준비됨 (실행 셀 사용 가능)" if client else "키 없음 — 실행 셀은 건너뜀")

## 관련 도메인 — 고객지원/주문운영 (결정적 데이터셋 + 도구 5개)

`find_customer → list_orders → get_charges → issue_refund → notify_customer` 로 이어지는 실제 워크플로.
데이터가 고정이라 정답을 정확히 계산할 수 있다. (ORD-1002 는 15,000원이 3번 청구된 '삼중 청구' 케이스)

In [ ]:
SUPPORT_DATA = {
    "customers": [
        {"id": "CUST-1", "name": "Sarah Chen",  "email": "sarah@example.com",  "tier": "gold"},
        {"id": "CUST-2", "name": "Minjun Park", "email": "minjun@example.com", "tier": "silver"},
    ],
    "orders": [
        {"id": "ORD-1001", "customer_id": "CUST-1", "status": "paid",      "total": 42000},
        {"id": "ORD-1002", "customer_id": "CUST-1", "status": "paid",      "total": 15000},
        {"id": "ORD-1003", "customer_id": "CUST-2", "status": "cancelled", "total": 30000},
        {"id": "ORD-1004", "customer_id": "CUST-2", "status": "paid",      "total": 8000},
    ],
    "charges": [
        {"id": "CHG-1", "order_id": "ORD-1001", "amount": 42000, "status": "succeeded"},
        {"id": "CHG-2", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-3", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-4", "order_id": "ORD-1002", "amount": 15000, "status": "succeeded"},
        {"id": "CHG-5", "order_id": "ORD-1003", "amount": 30000, "status": "refunded"},
        {"id": "CHG-6", "order_id": "ORD-1004", "amount":  8000, "status": "succeeded"},
    ],
}

def _find_customer(args):
    q = (args.get("query") or "").lower()
    if not q:
        return "<error>query 필요</error>"
    for c in SUPPORT_DATA["customers"]:
        if q in c["email"].lower() or q in c["name"].lower():
            return json.dumps(c, ensure_ascii=False)
    return json.dumps({"error": "not found", "query": args.get("query")}, ensure_ascii=False)

def _list_orders(args):
    cid, status = args.get("customer_id"), args.get("status")
    if not cid:
        return "<error>customer_id 필요</error>"
    out = [o for o in SUPPORT_DATA["orders"] if o["customer_id"] == cid and (not status or o["status"] == status)]
    return json.dumps(out, ensure_ascii=False)

def _get_charges(args):
    oid = args.get("order_id")
    if not oid:
        return "<error>order_id 필요</error>"
    return json.dumps([c for c in SUPPORT_DATA["charges"] if c["order_id"] == oid], ensure_ascii=False)

def _issue_refund(args):
    chg, reason = args.get("charge_id"), args.get("reason", "")
    if not chg:
        return "<error>charge_id 필요</error>"
    found = next((c for c in SUPPORT_DATA["charges"] if c["id"] == chg), None)
    if not found:
        return json.dumps({"error": "charge 없음", "charge_id": chg}, ensure_ascii=False)
    return json.dumps({"refund_id": "RFND-" + chg, "charge_id": chg, "amount": found["amount"],
                       "status": "refunded", "reason": reason}, ensure_ascii=False)

def _notify_customer(args):
    cid, ch, msg = args.get("customer_id"), args.get("channel"), args.get("message", "")
    if not cid or not ch:
        return "<error>customer_id, channel 필요</error>"
    return json.dumps({"notification_id": "NOTIF-" + cid, "channel": ch, "delivered": True, "chars": len(msg)}, ensure_ascii=False)

TOOLS_REL = [
    {"type": "function", "name": "find_customer",
     "description": "이메일 또는 이름으로 고객을 조회한다. 부분일치를 지원하며 첫 일치 고객의 id·이름·이메일·등급(tier)을 JSON 으로 반환한다. 다른 도구(list_orders 등)에 넣을 customer_id 를 얻는 출발점으로 쓴다.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "고객 이메일 또는 이름(부분일치). 예: 'sarah@example.com' 또는 'Sarah'"}}, "required": ["query"]}},
    {"type": "function", "name": "list_orders",
     "description": "특정 고객의 주문 목록을 반환한다. status 로 필터할 수 있다. 각 주문의 id·status·total(원)을 JSON 배열로 준다. customer_id 는 먼저 find_customer 로 확인하라.",
     "parameters": {"type": "object", "properties": {"customer_id": {"type": "string", "description": "고객 id. 예: 'CUST-1'"}, "status": {"type": "string", "enum": ["paid", "pending", "cancelled", "refunded"], "description": "주문 상태 필터(선택). 생략하면 전체"}}, "required": ["customer_id"]}},
    {"type": "function", "name": "get_charges",
     "description": "한 주문에 대한 결제(청구) 기록을 모두 반환한다. 각 청구의 id·amount·status 를 JSON 배열로 준다. 같은 주문에 succeeded 청구가 여러 건이면 중복(다중) 청구를 의심할 근거가 된다.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string", "description": "주문 id. 예: 'ORD-1002'"}}, "required": ["order_id"]}},
    {"type": "function", "name": "issue_refund",
     "description": "특정 청구(charge)를 환불 처리한다. 환불 확인 id(RFND- 로 시작)와 금액·상태를 반환한다. 중복 청구를 되돌릴 때 사용하며 charge_id 는 get_charges 결과에서 고른다.",
     "parameters": {"type": "object", "properties": {"charge_id": {"type": "string", "description": "환불할 청구 id. 예: 'CHG-3'"}, "reason": {"type": "string", "description": "환불 사유(짧게). 예: '중복 청구'"}}, "required": ["charge_id"]}},
    {"type": "function", "name": "notify_customer",
     "description": "고객에게 알림 메시지를 발송한다. 알림 id 와 전달 여부를 반환한다. 환불 처리 후 고객에게 결과를 알릴 때 사용한다.",
     "parameters": {"type": "object", "properties": {"customer_id": {"type": "string", "description": "고객 id. 예: 'CUST-1'"}, "channel": {"type": "string", "enum": ["email", "sms"], "description": "발송 채널"}, "message": {"type": "string", "description": "보낼 메시지 본문"}}, "required": ["customer_id", "channel", "message"]}},
]
IMPL_REL = {"find_customer": _find_customer, "list_orders": _list_orders, "get_charges": _get_charges,
            "issue_refund": _issue_refund, "notify_customer": _notify_customer}
print("관련 도구 5개 · 데이터: 고객", len(SUPPORT_DATA["customers"]), "주문", len(SUPPORT_DATA["orders"]), "청구", len(SUPPORT_DATA["charges"]))

## 방해 도구 15개 (하위 서브에이전트 3개가 병렬 작성)

전자상거래 지원과 **완전히 다른 컨텍스트**의 도구들 — 실제 프로덕션 API 문서처럼 촘촘히 작성됨.
네임스페이스를 부풀리는 노이즈다. 아래 세 셀은 각 서브에이전트 산출물을 그대로 담았다.

In [ ]:
import json

# =============================================================================
# ns_D1 — Distractor tool namespace
# 전자상거래 고객지원/주문운영과 완전히 무관한 5개 도메인의 mock 도구.
# 각 IMPL 함수는 결정적(deterministic) — 난수/네트워크/현재시각 사용 금지.
# =============================================================================


# ------------------------------------------------------------------ TOOLS -----

TOOLS = [
    {
        "type": "function",
        "name": "get_weather_forecast",
        "description": (
            "지정한 도시의 단기 일별 기상 예보를 조회한다. 여행/야외활동 계획이나 "
            "'내일 서울 날씨 어때?' 같은 질문에 답할 때 사용한다. 최고/최저 기온, "
            "강수확률, 하늘상태, 풍속을 일 단위로 최대 7일까지 반환하며, 단위계(섭씨/화씨)를 "
            "지정할 수 있다. 과거 관측이 아닌 미래 예보만 다룬다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "도시명. 한글 또는 영문. 예: 'Seoul', '서울', 'Tokyo'.",
                },
                "days": {
                    "type": "integer",
                    "description": "예보 일수. 1~7 사이 정수. 기본값 3. 오늘을 1일차로 센다.",
                    "minimum": 1,
                    "maximum": 7,
                },
                "units": {
                    "type": "string",
                    "description": "기온 단위계. 'metric'=섭씨, 'imperial'=화씨. 기본값 'metric'.",
                    "enum": ["metric", "imperial"],
                },
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "search_flights",
        "description": (
            "출발지-도착지 공항(IATA 3-letter 코드) 사이의 항공권을 검색한다. 특정 날짜의 "
            "직항/경유 노선을 가격 오름차순으로 반환하며, 좌석 등급과 성인 승객 수를 지정할 수 있다. "
            "'다음 주 인천에서 나리타 가는 항공권 찾아줘' 같은 여정 탐색에 사용한다. 실제 예약/결제는 "
            "수행하지 않고 조회만 한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {
                    "type": "string",
                    "description": "출발 공항 IATA 코드. 대문자 3글자. 예: 'ICN', 'GMP'.",
                },
                "destination": {
                    "type": "string",
                    "description": "도착 공항 IATA 코드. 대문자 3글자. 예: 'NRT', 'JFK'.",
                },
                "date": {
                    "type": "string",
                    "description": "출발 날짜. ISO 8601 형식 'YYYY-MM-DD'. 예: '2026-08-15'.",
                },
                "cabin_class": {
                    "type": "string",
                    "description": "좌석 등급. 기본값 'economy'.",
                    "enum": ["economy", "premium_economy", "business", "first"],
                },
                "adults": {
                    "type": "integer",
                    "description": "성인 승객 수. 1~9 사이 정수. 기본값 1.",
                    "minimum": 1,
                    "maximum": 9,
                },
            },
            "required": ["origin", "destination", "date"],
        },
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": (
            "한 통화 금액을 다른 통화로 환산한다. ISO 4217 통화 코드를 사용하며 환율, 환산 금액, "
            "기준 시각을 함께 반환한다. '100달러는 몇 원이야?' 또는 여행 예산 계산 같은 상황에 사용한다. "
            "송금/결제는 하지 않고 참고용 환산값만 제공한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "환산할 금액. 0보다 큰 실수. 예: 100, 49.99.",
                    "exclusiveMinimum": 0,
                },
                "from_currency": {
                    "type": "string",
                    "description": "원본 통화 ISO 4217 코드. 대문자 3글자. 예: 'USD', 'KRW', 'EUR', 'JPY', 'GBP'.",
                },
                "to_currency": {
                    "type": "string",
                    "description": "목표 통화 ISO 4217 코드. 대문자 3글자. 예: 'KRW', 'USD'.",
                },
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    {
        "type": "function",
        "name": "lookup_dns_records",
        "description": (
            "도메인의 DNS 레코드를 조회한다. A/AAAA/MX/TXT/CNAME/NS 등 레코드 타입별로 값과 TTL(초)을 "
            "반환한다. '도메인 메일 서버(MX) 확인' 또는 'A 레코드 IP 조회' 같은 네트워크/인프라 점검에 사용한다. "
            "레코드를 변경하지 않는 읽기 전용 조회다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "domain": {
                    "type": "string",
                    "description": "조회할 도메인명. 프로토콜 없이. 예: 'example.com', 'mail.google.com'.",
                },
                "record_type": {
                    "type": "string",
                    "description": "DNS 레코드 타입. 기본값 'A'.",
                    "enum": ["A", "AAAA", "MX", "TXT", "CNAME", "NS"],
                },
            },
            "required": ["domain"],
        },
    },
    {
        "type": "function",
        "name": "translate_text",
        "description": (
            "입력 텍스트를 목표 언어로 번역한다. 원본 언어는 지정하거나 'auto'로 자동 감지할 수 있고, "
            "감지된 소스 언어와 번역문을 함께 반환한다. '이 문장 영어로 번역해줘' 같은 다국어 변환 요청에 사용한다. "
            "언어 코드는 ISO 639-1 2글자 소문자를 쓴다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "번역할 원문. 최대 5000자. 예: '안녕하세요, 반갑습니다.'",
                },
                "target_lang": {
                    "type": "string",
                    "description": "목표 언어 ISO 639-1 코드(소문자 2글자). 예: 'en', 'ko', 'ja', 'zh', 'es', 'fr', 'de'.",
                    "enum": ["en", "ko", "ja", "zh", "es", "fr", "de"],
                },
                "source_lang": {
                    "type": "string",
                    "description": "원본 언어 ISO 639-1 코드 또는 'auto'(자동 감지). 기본값 'auto'.",
                },
            },
            "required": ["text", "target_lang"],
        },
    },
]


# ------------------------------------------------------------------- IMPL -----

def _get_weather_forecast(args: dict) -> str:
    import hashlib

    city = args.get("city")
    if not city:
        return "error: 'city' is required"
    days = args.get("days", 3)
    try:
        days = int(days)
    except (TypeError, ValueError):
        return "error: 'days' must be an integer"
    if days < 1 or days > 7:
        return "error: 'days' must be between 1 and 7"
    units = args.get("units", "metric")
    if units not in ("metric", "imperial"):
        return "error: 'units' must be 'metric' or 'imperial'"

    # 결정적 시드: 도시명 해시로 기준 기온/상태를 유도
    seed = int(hashlib.md5(city.strip().lower().encode("utf-8")).hexdigest(), 16)
    base_high_c = 8 + (seed % 22)          # 8~29도
    sky_states = ["맑음", "구름조금", "흐림", "비", "소나기", "눈"]
    unit_label = "C" if units == "metric" else "F"

    def to_unit(c):
        return c if units == "metric" else round(c * 9 / 5 + 32)

    forecast = []
    for d in range(days):
        # 날짜별 결정적 변동
        wobble = ((seed >> (d + 1)) % 7) - 3     # -3~+3
        high_c = base_high_c + wobble
        low_c = high_c - (5 + ((seed >> d) % 4))  # 최고 대비 5~8도 낮음
        pop = ((seed >> (d + 2)) % 10) * 10       # 0~90 %
        sky = sky_states[(seed >> (d + 3)) % len(sky_states)]
        wind = 1 + ((seed >> (d + 4)) % 12)       # 1~12 m/s
        forecast.append({
            "day_offset": d,
            "high": to_unit(high_c),
            "low": to_unit(low_c),
            "unit": unit_label,
            "sky": sky,
            "precip_probability_pct": pop,
            "wind_speed_ms": wind,
        })

    return json.dumps({
        "city": city,
        "units": units,
        "days": days,
        "forecast": forecast,
    }, ensure_ascii=False)


def _search_flights(args: dict) -> str:
    import hashlib

    origin = args.get("origin")
    destination = args.get("destination")
    date = args.get("date")
    if not origin:
        return "error: 'origin' is required"
    if not destination:
        return "error: 'destination' is required"
    if not date:
        return "error: 'date' is required"
    cabin_class = args.get("cabin_class", "economy")
    if cabin_class not in ("economy", "premium_economy", "business", "first"):
        return "error: invalid 'cabin_class'"
    adults = args.get("adults", 1)
    try:
        adults = int(adults)
    except (TypeError, ValueError):
        return "error: 'adults' must be an integer"
    if adults < 1 or adults > 9:
        return "error: 'adults' must be between 1 and 9"

    o = origin.strip().upper()
    d = destination.strip().upper()
    seed = int(hashlib.md5(f"{o}-{d}-{date}".encode("utf-8")).hexdigest(), 16)

    carriers = [
        ("KE", "Korean Air"),
        ("OZ", "Asiana Airlines"),
        ("JL", "Japan Airlines"),
        ("DL", "Delta Air Lines"),
        ("SQ", "Singapore Airlines"),
    ]
    cabin_multiplier = {
        "economy": 1.0,
        "premium_economy": 1.8,
        "business": 3.2,
        "first": 5.5,
    }[cabin_class]
    base_fare = 120 + (seed % 380)   # 120~499 USD 기준 편도

    results = []
    for i in range(3):
        code, name = carriers[(seed >> (i * 2)) % len(carriers)]
        flight_no = 100 + ((seed >> (i + 1)) % 900)
        dep_hour = (6 + ((seed >> (i + 2)) % 15))     # 06~20시
        dur_min = 90 + ((seed >> (i + 3)) % 600)      # 1.5~11.5시간
        stops = (seed >> (i + 4)) % 2                 # 0 or 1
        fare_each = round((base_fare + i * 45 + stops * 60) * cabin_multiplier, 2)
        arr_total = dep_hour * 60 + dur_min
        results.append({
            "carrier": name,
            "flight_number": f"{code}{flight_no}",
            "origin": o,
            "destination": d,
            "date": date,
            "departure_time": f"{dep_hour:02d}:{((seed >> i) % 6) * 10:02d}",
            "arrival_time": f"{(arr_total // 60) % 24:02d}:{arr_total % 60:02d}",
            "duration_minutes": dur_min,
            "stops": stops,
            "cabin_class": cabin_class,
            "fare_per_adult_usd": fare_each,
            "total_fare_usd": round(fare_each * adults, 2),
        })

    results.sort(key=lambda r: r["fare_per_adult_usd"])
    return json.dumps({
        "origin": o,
        "destination": d,
        "date": date,
        "adults": adults,
        "cabin_class": cabin_class,
        "currency": "USD",
        "results": results,
    }, ensure_ascii=False)


def _convert_currency(args: dict) -> str:
    amount = args.get("amount")
    from_currency = args.get("from_currency")
    to_currency = args.get("to_currency")
    if amount is None:
        return "error: 'amount' is required"
    if from_currency is None:
        return "error: 'from_currency' is required"
    if to_currency is None:
        return "error: 'to_currency' is required"
    try:
        amount = float(amount)
    except (TypeError, ValueError):
        return "error: 'amount' must be a number"
    if amount <= 0:
        return "error: 'amount' must be greater than 0"

    src = from_currency.strip().upper()
    dst = to_currency.strip().upper()

    # USD 기준 고정 환율표 (1 USD = N 통화). 결정적.
    usd_rates = {
        "USD": 1.0,
        "KRW": 1350.0,
        "EUR": 0.92,
        "JPY": 155.0,
        "GBP": 0.79,
        "CNY": 7.24,
        "AUD": 1.52,
        "CAD": 1.36,
    }
    if src not in usd_rates:
        return f"error: unsupported from_currency '{src}'"
    if dst not in usd_rates:
        return f"error: unsupported to_currency '{dst}'"

    # src -> USD -> dst
    rate = usd_rates[dst] / usd_rates[src]
    converted = round(amount * rate, 4)
    return json.dumps({
        "amount": amount,
        "from_currency": src,
        "to_currency": dst,
        "rate": round(rate, 6),
        "converted_amount": converted,
        "as_of": "2026-01-02T00:00:00Z",
        "note": "reference rate (mock, non-transactional)",
    }, ensure_ascii=False)


def _lookup_dns_records(args: dict) -> str:
    import hashlib

    domain = args.get("domain")
    if not domain:
        return "error: 'domain' is required"
    record_type = args.get("record_type", "A")
    if record_type not in ("A", "AAAA", "MX", "TXT", "CNAME", "NS"):
        return f"error: unsupported record_type '{record_type}'"

    d = domain.strip().lower().lstrip(".")
    seed = int(hashlib.md5(d.encode("utf-8")).hexdigest(), 16)

    records = []
    if record_type == "A":
        oct2 = (seed >> 4) % 256
        oct3 = (seed >> 8) % 256
        oct4 = (seed >> 12) % 254 + 1
        records.append({"value": f"93.{oct2}.{oct3}.{oct4}", "ttl": 300})
        records.append({"value": f"104.{(oct2 + 7) % 256}.{oct3}.{(oct4 + 11) % 254 + 1}", "ttl": 300})
    elif record_type == "AAAA":
        h = hashlib.md5(d.encode("utf-8")).hexdigest()
        groups = [h[i:i + 4] for i in range(0, 16, 4)]
        records.append({"value": "2606:4700:" + ":".join(groups), "ttl": 300})
    elif record_type == "MX":
        records.append({"value": f"aspmx.l.{d}", "priority": 10, "ttl": 3600})
        records.append({"value": f"alt1.aspmx.l.{d}", "priority": 20, "ttl": 3600})
    elif record_type == "TXT":
        token = hashlib.md5(("txt" + d).encode("utf-8")).hexdigest()[:32]
        records.append({"value": "v=spf1 include:_spf." + d + " ~all", "ttl": 3600})
        records.append({"value": f"google-site-verification={token}", "ttl": 3600})
    elif record_type == "CNAME":
        records.append({"value": f"{d}.cdn.example-edge.net", "ttl": 1800})
    elif record_type == "NS":
        for i in range(1, 3):
            records.append({"value": f"ns{i}.{d}", "ttl": 86400})

    return json.dumps({
        "domain": d,
        "record_type": record_type,
        "records": records,
        "authoritative": True,
    }, ensure_ascii=False)


def _translate_text(args: dict) -> str:
    text = args.get("text")
    target_lang = args.get("target_lang")
    if not text:
        return "error: 'text' is required"
    if not target_lang:
        return "error: 'target_lang' is required"
    supported = ("en", "ko", "ja", "zh", "es", "fr", "de")
    if target_lang not in supported:
        return f"error: unsupported target_lang '{target_lang}'"
    source_lang = args.get("source_lang", "auto")
    if len(text) > 5000:
        return "error: 'text' exceeds 5000 characters"

    # 결정적 언어 감지: 문자 범위로 스크립트 추정
    def detect(s):
        for ch in s:
            code = ord(ch)
            if 0xAC00 <= code <= 0xD7A3 or 0x1100 <= code <= 0x11FF:
                return "ko"
            if 0x3040 <= code <= 0x30FF:
                return "ja"
            if 0x4E00 <= code <= 0x9FFF:
                return "zh"
        return "en"

    detected = detect(text) if source_lang == "auto" else source_lang

    lang_names = {
        "en": "English", "ko": "Korean", "ja": "Japanese", "zh": "Chinese",
        "es": "Spanish", "fr": "French", "de": "German",
    }
    tgt_name = lang_names.get(target_lang, target_lang)

    if detected == target_lang:
        translated = text
    else:
        # mock 번역: 원문을 보존하되 목표 언어 태그를 붙인 결정적 문자열
        translated = f"[{tgt_name}] {text}"

    return json.dumps({
        "source_lang": detected,
        "target_lang": target_lang,
        "original_text": text,
        "translated_text": translated,
        "char_count": len(text),
    }, ensure_ascii=False)


IMPL = {
    "get_weather_forecast": _get_weather_forecast,
    "search_flights": _search_flights,
    "convert_currency": _convert_currency,
    "lookup_dns_records": _lookup_dns_records,
    "translate_text": _translate_text,
}

# ── 그룹 D1 캡처 ──
TOOLS_D1, IMPL_D1 = TOOLS, IMPL
print("D1(방해) 도구:", [t["name"] for t in TOOLS_D1])


In [ ]:
import json


# ---------------------------------------------------------------------------
# Namespace D2 — distractor tools (전자상거래 CS/주문운영과 무관한 5개 도메인)
#   1) analyze_food_nutrition   : 영양/칼로리 분석
#   2) generate_chord_progression: 음악 이론 (코드 진행 생성)
#   3) get_soil_moisture_reading : 토양 수분 IoT 센서 조회 (스마트팜)
#   4) get_moon_phase            : 천문 (특정 날짜의 달 위상)
#   5) calculate_income_tax      : 소득세 구간 계산
# ---------------------------------------------------------------------------


# ============================ 1. 영양/칼로리 분석 ============================
def _impl_analyze_food_nutrition(args: dict) -> str:
    food = args.get("food_name")
    if not food or not str(food).strip():
        return "error: 'food_name' 은 필수입니다."
    grams = args.get("serving_grams", 100)
    try:
        grams = float(grams)
    except (TypeError, ValueError):
        return "error: 'serving_grams' 는 숫자여야 합니다."
    if grams <= 0:
        return "error: 'serving_grams' 는 0 보다 커야 합니다."

    # 결정적 mock DB (100g 기준). 미등록 음식은 이름 해시로 그럴듯한 고정값 유도.
    db = {
        "백미밥":       {"kcal": 130.0, "carb_g": 28.1, "protein_g": 2.7, "fat_g": 0.3, "sodium_mg": 1.0},
        "닭가슴살":     {"kcal": 165.0, "carb_g": 0.0,  "protein_g": 31.0, "fat_g": 3.6, "sodium_mg": 74.0},
        "바나나":       {"kcal": 89.0,  "carb_g": 22.8, "protein_g": 1.1,  "fat_g": 0.3, "sodium_mg": 1.0},
        "고구마":       {"kcal": 86.0,  "carb_g": 20.1, "protein_g": 1.6,  "fat_g": 0.1, "sodium_mg": 55.0},
        "달걀":         {"kcal": 155.0, "carb_g": 1.1,  "protein_g": 13.0, "fat_g": 11.0, "sodium_mg": 124.0},
        "아몬드":       {"kcal": 579.0, "carb_g": 21.6, "protein_g": 21.2, "fat_g": 49.9, "sodium_mg": 1.0},
    }
    key = str(food).strip()
    if key in db:
        base = db[key]
        source = "verified"
    else:
        # 이름에서 유도한 결정적 pseudo 값 (난수 아님).
        h = 0
        for ch in key:
            h = (h * 31 + ord(ch)) % 100000
        base = {
            "kcal":       round(120 + (h % 380), 1),
            "carb_g":     round((h % 55), 1),
            "protein_g":  round(((h // 7) % 30), 1),
            "fat_g":      round(((h // 13) % 25), 1),
            "sodium_mg":  round(((h // 3) % 900), 1),
        }
        source = "estimated"

    factor = grams / 100.0
    scaled = {k: round(v * factor, 1) for k, v in base.items()}
    result = {
        "food_name": key,
        "serving_grams": round(grams, 1),
        "source": source,
        "nutrition": {
            "energy_kcal": scaled["kcal"],
            "carbohydrate_g": scaled["carb_g"],
            "protein_g": scaled["protein_g"],
            "fat_g": scaled["fat_g"],
            "sodium_mg": scaled["sodium_mg"],
        },
        "per_100g_basis": base,
    }
    return json.dumps(result, ensure_ascii=False)


# ============================ 2. 음악 이론 (코드 진행) ============================
def _impl_generate_chord_progression(args: dict) -> str:
    key = args.get("key")
    if not key or not str(key).strip():
        return "error: 'key' 는 필수입니다. 예) 'C', 'A'."
    mode = args.get("mode", "major")
    if mode not in ("major", "minor"):
        return "error: 'mode' 는 'major' 또는 'minor' 여야 합니다."
    length = args.get("length", 4)
    try:
        length = int(length)
    except (TypeError, ValueError):
        return "error: 'length' 는 정수여야 합니다."
    if length < 1 or length > 8:
        return "error: 'length' 는 1~8 사이여야 합니다."

    chromatic = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    enharm = {"Db": "C#", "Eb": "D#", "Gb": "F#", "Ab": "G#", "Bb": "A#"}
    root = str(key).strip()
    root = enharm.get(root, root)
    if root not in chromatic:
        return "error: 알 수 없는 key '%s'. C, C#, D ... B (또는 Db, Eb ...) 중 하나여야 합니다." % key

    root_idx = chromatic.index(root)

    if mode == "major":
        # 장음계 도수 간격 및 각 도수의 코드 퀄리티
        steps = [0, 2, 4, 5, 7, 9, 11]
        qualities = ["", "m", "m", "", "", "m", "dim"]
        roman = ["I", "ii", "iii", "IV", "V", "vi", "vii°"]
        preset = [0, 4, 5, 3, 1, 5, 3, 4]  # I-V-vi-IV ... 결정적 순환
    else:
        # 자연 단음계
        steps = [0, 2, 3, 5, 7, 8, 10]
        qualities = ["m", "dim", "", "m", "m", "", ""]
        roman = ["i", "ii°", "III", "iv", "v", "VI", "VII"]
        preset = [0, 5, 2, 6, 3, 4, 0, 5]  # i-VI-III-VII ... 결정적 순환

    scale_notes = [chromatic[(root_idx + s) % 12] for s in steps]

    chords = []
    for i in range(length):
        deg = preset[i]
        note = scale_notes[deg]
        chord_name = note + qualities[deg]
        chords.append({
            "degree": roman[deg],
            "chord": chord_name,
            "root_note": note,
        })

    result = {
        "key": root,
        "mode": mode,
        "length": length,
        "scale_notes": scale_notes,
        "progression": [c["chord"] for c in chords],
        "roman_numerals": [c["degree"] for c in chords],
        "detail": chords,
    }
    return json.dumps(result, ensure_ascii=False)


# ============================ 3. 토양 수분 IoT 센서 조회 ============================
def _impl_get_soil_moisture_reading(args: dict) -> str:
    sensor_id = args.get("sensor_id")
    if not sensor_id or not str(sensor_id).strip():
        return "error: 'sensor_id' 는 필수입니다. 예) 'FARM-A-07'."
    depth = args.get("depth_cm", 10)
    try:
        depth = int(depth)
    except (TypeError, ValueError):
        return "error: 'depth_cm' 는 정수여야 합니다."
    if depth not in (10, 20, 30, 40):
        return "error: 'depth_cm' 는 10, 20, 30, 40 중 하나여야 합니다."
    unit = args.get("unit", "vwc_percent")
    if unit not in ("vwc_percent", "kpa"):
        return "error: 'unit' 는 'vwc_percent' 또는 'kpa' 여야 합니다."

    sid = str(sensor_id).strip()
    # 센서 ID 에서 유도한 결정적 base 값.
    h = 0
    for ch in sid:
        h = (h * 37 + ord(ch)) % 100000
    # 깊을수록 수분 함량이 조금 더 높게(관수 후 하강 지연) 나오는 그럴듯한 모델.
    base_vwc = 18.0 + (h % 220) / 10.0          # 18.0 ~ 39.9 %
    vwc = round(base_vwc + (depth - 10) * 0.6, 1)
    if vwc > 55.0:
        vwc = 55.0
    # VWC(%) -> 장력(kPa) 근사: 습할수록 장력 낮음.
    tension_kpa = round(max(2.0, 120.0 - vwc * 2.4), 1)

    if vwc < 20.0:
        status = "dry"
        advice = "관수 필요: 근권 수분이 위조점에 근접."
    elif vwc < 35.0:
        status = "optimal"
        advice = "적정 범위: 추가 관수 불필요."
    else:
        status = "wet"
        advice = "과습 주의: 배수 상태 점검 권장."

    reading_value = vwc if unit == "vwc_percent" else tension_kpa
    result = {
        "sensor_id": sid,
        "depth_cm": depth,
        "unit": unit,
        "value": reading_value,
        "vwc_percent": vwc,
        "tension_kpa": tension_kpa,
        "soil_temperature_c": round(16.0 + (h % 90) / 10.0, 1),
        "battery_percent": 70 + (h % 30),
        "status": status,
        "advice": advice,
        "sample_window": "last_15min_avg",
    }
    return json.dumps(result, ensure_ascii=False)


# ============================ 4. 천문 (달 위상) ============================
def _impl_get_moon_phase(args: dict) -> str:
    date = args.get("date")
    if not date or not str(date).strip():
        return "error: 'date' 는 필수입니다. 형식 'YYYY-MM-DD'."
    d = str(date).strip()
    parts = d.split("-")
    if len(parts) != 3:
        return "error: 'date' 형식은 'YYYY-MM-DD' 여야 합니다."
    try:
        y, mo, da = int(parts[0]), int(parts[1]), int(parts[2])
    except ValueError:
        return "error: 'date' 의 연/월/일은 정수여야 합니다."
    if not (1 <= mo <= 12) or not (1 <= da <= 31):
        return "error: 'date' 의 월(1-12)/일(1-31) 범위가 올바르지 않습니다."

    # 결정적 삭망 계산: 알려진 신월 기준(2000-01-06) 이후 경과일 / 삭망월(29.53059일).
    # 그레고리력 -> 율리우스 적일(정수) 근사.
    a = (14 - mo) // 12
    yy = y + 4800 - a
    mm = mo + 12 * a - 3
    jdn = da + (153 * mm + 2) // 5 + 365 * yy + yy // 4 - yy // 100 + yy // 400 - 32045
    known_new_moon_jdn = 2451550  # 2000-01-06 근처
    synodic = 29.53058867
    days = (jdn - known_new_moon_jdn) % synodic
    if days < 0:
        days += synodic

    illum = round((1 - abs(1 - (days / synodic) * 2)) * 100.0, 1)  # 0=삭, 100=망
    age = round(days, 2)

    if days < 1.84566:
        phase, emoji = "신월(New Moon)", "🌑"
    elif days < 5.53699:
        phase, emoji = "초승달(Waxing Crescent)", "🌒"
    elif days < 9.22831:
        phase, emoji = "상현(First Quarter)", "🌓"
    elif days < 12.91963:
        phase, emoji = "상현망간(Waxing Gibbous)", "🌔"
    elif days < 16.61096:
        phase, emoji = "보름달(Full Moon)", "🌕"
    elif days < 20.30228:
        phase, emoji = "하현망간(Waning Gibbous)", "🌖"
    elif days < 23.99361:
        phase, emoji = "하현(Last Quarter)", "🌗"
    elif days < 27.68493:
        phase, emoji = "그믐달(Waning Crescent)", "🌘"
    else:
        phase, emoji = "신월(New Moon)", "🌑"

    result = {
        "date": d,
        "phase_name": phase,
        "emoji": emoji,
        "moon_age_days": age,
        "illumination_percent": illum,
        "synodic_month_days": synodic,
        "julian_day_number": jdn,
    }
    return json.dumps(result, ensure_ascii=False)


# ============================ 5. 소득세 구간 계산 ============================
def _impl_calculate_income_tax(args: dict) -> str:
    income = args.get("annual_income")
    if income is None:
        return "error: 'annual_income' 은 필수입니다 (연 과세표준, 원 단위)."
    try:
        income = float(income)
    except (TypeError, ValueError):
        return "error: 'annual_income' 은 숫자여야 합니다."
    if income < 0:
        return "error: 'annual_income' 은 0 이상이어야 합니다."

    deduction = args.get("deduction", 0)
    try:
        deduction = float(deduction)
    except (TypeError, ValueError):
        return "error: 'deduction' 은 숫자여야 합니다."
    if deduction < 0:
        return "error: 'deduction' 은 0 이상이어야 합니다."
    include_local = bool(args.get("include_local_tax", True))

    taxable = max(0.0, income - deduction)

    # 한국 종합소득세 누진세율 구간 (하한, 상한, 세율, 누진공제)
    brackets = [
        (0,          14_000_000,   0.06, 0),
        (14_000_000, 50_000_000,   0.15, 1_260_000),
        (50_000_000, 88_000_000,   0.24, 5_760_000),
        (88_000_000, 150_000_000,  0.35, 15_440_000),
        (150_000_000,300_000_000,  0.38, 19_940_000),
        (300_000_000,500_000_000,  0.40, 25_940_000),
        (500_000_000,1_000_000_000,0.42, 35_940_000),
        (1_000_000_000, float("inf"), 0.45, 65_940_000),
    ]

    marginal_rate = 0.06
    quick_deduction = 0
    for lo, hi, rate, qd in brackets:
        if taxable > lo:
            marginal_rate = rate
            quick_deduction = qd
        if lo < taxable <= hi:
            break

    income_tax = max(0.0, taxable * marginal_rate - quick_deduction)
    income_tax = round(income_tax)
    local_tax = round(income_tax * 0.10) if include_local else 0
    total_tax = income_tax + local_tax
    effective_rate = round((total_tax / income * 100.0), 2) if income > 0 else 0.0

    result = {
        "annual_income": round(income),
        "deduction": round(deduction),
        "taxable_income": round(taxable),
        "marginal_rate": marginal_rate,
        "progressive_deduction": quick_deduction,
        "income_tax": income_tax,
        "local_income_tax": local_tax,
        "total_tax": total_tax,
        "effective_rate_percent": effective_rate,
        "currency": "KRW",
        "include_local_tax": include_local,
    }
    return json.dumps(result, ensure_ascii=False)


# ---------------------------------------------------------------------------
# TOOLS: OpenAI Responses function-tool 정의
# ---------------------------------------------------------------------------
TOOLS = [
    {
        "type": "function",
        "name": "analyze_food_nutrition",
        "description": (
            "음식 이름과 섭취 중량(g)을 받아 해당 분량의 영양성분(열량, 탄수화물, 단백질, 지방, "
            "나트륨)을 분석해 반환한다. 식단 기록, 칼로리 계산, 다이어트/운동 코칭처럼 특정 음식의 "
            "영양 정보를 알아야 할 때 사용한다. 내부 등록 음식은 검증값(source=verified)을, "
            "미등록 음식은 이름 기반 추정값(source=estimated)을 제공하며, 모든 수치는 입력 중량에 "
            "비례해 스케일링된다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "food_name": {
                    "type": "string",
                    "description": "분석할 음식 이름. 한국어 일반명 권장. 예) '백미밥', '닭가슴살', '바나나'.",
                },
                "serving_grams": {
                    "type": "number",
                    "description": "섭취 중량(g). 양수. 미지정 시 100g 기준으로 계산. 예) 150.",
                },
            },
            "required": ["food_name"],
        },
    },
    {
        "type": "function",
        "name": "generate_chord_progression",
        "description": (
            "지정한 조성(key)과 장/단조(mode)에 맞는 다이어토닉 코드 진행을 생성한다. 작곡/편곡 "
            "보조, 코드 연습, 반주 스케치가 필요할 때 사용한다. 스케일 구성음과 로마 숫자 화성도수를 "
            "함께 제공하며, length(1~8)만큼의 코드를 결정적 순환 패턴(장조 I-V-vi-IV 계열, 단조 "
            "i-VI-III-VII 계열)으로 뽑아준다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description": "으뜸음(조성). 예) 'C', 'G', 'A', 'F#', 'Bb'. 샾/플랫 표기 모두 허용.",
                },
                "mode": {
                    "type": "string",
                    "description": "장조/단조 선택. 미지정 시 'major'.",
                    "enum": ["major", "minor"],
                },
                "length": {
                    "type": "integer",
                    "description": "생성할 코드 개수(1~8). 미지정 시 4.",
                },
            },
            "required": ["key"],
        },
    },
    {
        "type": "function",
        "name": "get_soil_moisture_reading",
        "description": (
            "스마트팜에 설치된 토양 수분 IoT 센서의 최근 측정값을 조회한다. 관수 여부 판단, 근권 "
            "수분 모니터링, 과습/건조 경보 확인이 필요할 때 사용한다. 지정한 매설 깊이(depth_cm)의 "
            "체적수분함량(VWC %) 또는 수분장력(kPa)을 반환하며, 토양 온도·배터리 잔량과 함께 "
            "dry/optimal/wet 상태 및 관수 권고를 제공한다. 값은 최근 15분 평균 기준이다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "sensor_id": {
                    "type": "string",
                    "description": "센서 식별자. 예) 'FARM-A-07', 'ZONE3-SOIL-12'.",
                },
                "depth_cm": {
                    "type": "integer",
                    "description": "측정 매설 깊이(cm). 미지정 시 10.",
                    "enum": [10, 20, 30, 40],
                },
                "unit": {
                    "type": "string",
                    "description": "반환 단위. 'vwc_percent'(체적수분함량 %) 또는 'kpa'(수분장력). 미지정 시 'vwc_percent'.",
                    "enum": ["vwc_percent", "kpa"],
                },
            },
            "required": ["sensor_id"],
        },
    },
    {
        "type": "function",
        "name": "get_moon_phase",
        "description": (
            "특정 날짜(그레고리력)의 달 위상을 계산해 반환한다. 천문 관측 계획, 캘린더/위젯 표시, "
            "밤낚시·촬영 등 달빛 조건 확인에 사용한다. 삭망월(29.53일) 주기를 기준으로 달의 나이(일), "
            "조도(%), 위상명(신월/초승달/상현/보름 등)과 이모지, 율리우스 적일을 제공한다. 시간대는 "
            "고려하지 않고 날짜 단위로 결정적으로 계산한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "date": {
                    "type": "string",
                    "description": "조회 날짜. 형식 'YYYY-MM-DD'. 예) '2026-07-24'.",
                },
            },
            "required": ["date"],
        },
    },
    {
        "type": "function",
        "name": "calculate_income_tax",
        "description": (
            "연 과세표준(원)을 받아 한국 종합소득세 누진세율 구간에 따라 산출세액을 계산한다. 연말정산 "
            "예상세액 확인, 재무 계획, 세후 소득 추정에 사용한다. 공제액(deduction)을 반영한 과세표준에 "
            "8단계 누진세율(6%~45%)과 누진공제를 적용하며, 옵션으로 지방소득세(소득세의 10%)를 포함해 "
            "총부담세액과 실효세율(%)까지 반환한다. 통화 단위는 원(KRW)이다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "annual_income": {
                    "type": "number",
                    "description": "연 소득/과세표준(원 단위, 0 이상). 예) 55000000.",
                },
                "deduction": {
                    "type": "number",
                    "description": "각종 공제 합계(원). 과세표준에서 차감. 미지정 시 0. 예) 1500000.",
                },
                "include_local_tax": {
                    "type": "boolean",
                    "description": "지방소득세(산출세액의 10%) 포함 여부. 미지정 시 true.",
                },
            },
            "required": ["annual_income"],
        },
    },
]


# ---------------------------------------------------------------------------
# IMPL: 도구명 -> 실행 함수 매핑
# ---------------------------------------------------------------------------
IMPL = {
    "analyze_food_nutrition": _impl_analyze_food_nutrition,
    "generate_chord_progression": _impl_generate_chord_progression,
    "get_soil_moisture_reading": _impl_get_soil_moisture_reading,
    "get_moon_phase": _impl_get_moon_phase,
    "calculate_income_tax": _impl_calculate_income_tax,
}

# ── 그룹 D2 캡처 ──
TOOLS_D2, IMPL_D2 = TOOLS, IMPL
print("D2(방해) 도구:", [t["name"] for t in TOOLS_D2])


In [ ]:
import json

TOOLS = [
    {
        "type": "function",
        "name": "suggest_ingredient_substitute",
        "description": (
            "레시피에서 특정 재료가 없거나 알레르기/식이제한으로 사용할 수 없을 때 대체 가능한 재료와 "
            "환산 비율을 추천한다. 요리 종류(베이킹/볶음/소스 등)와 식이 제약을 고려해 최대 3개의 후보를 "
            "우선순위 순으로 돌려주며, 각 후보에 대해 원재료 대비 사용 비율과 맛/식감 변화 주의사항을 제공한다. "
            "재료를 실제로 구매하기 전 대체안을 빠르게 확인하고 싶을 때 사용한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "ingredient": {
                    "type": "string",
                    "description": "대체하려는 원재료명(한국어 또는 영어). 예: 'butter', '버터', 'buttermilk'",
                },
                "quantity": {
                    "type": "string",
                    "description": "원재료의 양. 단위 포함 문자열. 예: '200g', '1 cup', '2 tbsp'. 생략 시 비율만 계산.",
                },
                "dish_type": {
                    "type": "string",
                    "description": "요리 유형. 대체재 선정 기준이 됨.",
                    "enum": ["baking", "frying", "sauce", "soup", "salad", "general"],
                },
                "dietary_restriction": {
                    "type": "string",
                    "description": "식이 제약. 해당 제약을 위반하는 후보는 제외됨.",
                    "enum": ["none", "vegan", "gluten_free", "dairy_free", "nut_free"],
                },
            },
            "required": ["ingredient"],
        },
    },
    {
        "type": "function",
        "name": "git_blame_line",
        "description": (
            "Git 저장소에서 특정 파일의 특정 라인(또는 라인 범위)을 마지막으로 수정한 커밋과 작성자 정보를 "
            "조회한다. 코드 리뷰나 회귀 버그 추적 중 '이 줄을 누가/언제/왜 바꿨는지' 확인할 때 사용한다. "
            "반환값에는 커밋 해시(축약), 작성자, 커밋 시각(ISO8601), 커밋 요약 메시지, 해당 라인 스니펫이 포함된다. "
            "line_end 를 주면 범위 blame, 생략하면 단일 라인 blame 을 수행한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "repo": {
                    "type": "string",
                    "description": "저장소 식별자. 'owner/name' 형식. 예: 'acme/payments-api'",
                },
                "file_path": {
                    "type": "string",
                    "description": "저장소 루트 기준 상대 파일 경로. 예: 'src/utils/auth.py'",
                },
                "line_start": {
                    "type": "integer",
                    "description": "blame 을 조회할 시작 라인 번호(1-based). 예: 42",
                },
                "line_end": {
                    "type": "integer",
                    "description": "범위 blame 종료 라인 번호(포함). 생략 시 line_start 단일 라인만 조회. 예: 48",
                },
                "ref": {
                    "type": "string",
                    "description": "조회 기준 브랜치/태그/커밋. 기본값 'HEAD'. 예: 'main', 'v2.3.0'",
                },
            },
            "required": ["repo", "file_path", "line_start"],
        },
    },
    {
        "type": "function",
        "name": "format_legal_citation",
        "description": (
            "판례/법령의 서지 요소를 받아 지정한 인용 스타일에 맞는 정식 인용 문자열을 생성한다. 미국식 "
            "Bluebook, 한국 대법원 표기, APA 법률 인용 등 스타일별 규칙(사건명 이탤릭 여부, 권/면 표기 순서, "
            "연도 괄호 위치)을 적용한다. 준비서면·논문·메모에 넣을 인용을 표준 포맷으로 정리할 때 사용한다. "
            "pincite(특정 인용 면)를 주면 대표 면 뒤에 세부 면을 덧붙인다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "case_name": {
                    "type": "string",
                    "description": "사건명. 예: 'Roe v. Wade' 또는 '대법원 2011다12345'",
                },
                "reporter": {
                    "type": "string",
                    "description": "판례집/리포터 약어. 예: 'U.S.', 'F.3d', '집' (판례집)",
                },
                "volume": {
                    "type": "string",
                    "description": "권(volume) 번호. 예: '410'",
                },
                "page": {
                    "type": "string",
                    "description": "판례가 시작하는 면 번호. 예: '113'",
                },
                "year": {
                    "type": "string",
                    "description": "선고 연도(4자리). 예: '1973'",
                },
                "pincite": {
                    "type": "string",
                    "description": "특정 인용 면(선택). 예: '116'. 생략 가능.",
                },
                "style": {
                    "type": "string",
                    "description": "인용 스타일.",
                    "enum": ["bluebook", "korean_supreme", "apa"],
                },
            },
            "required": ["case_name", "reporter", "volume", "page", "year"],
        },
    },
    {
        "type": "function",
        "name": "convert_unit",
        "description": (
            "길이/무게/온도/부피 등 측정 단위를 서로 변환한다. 값과 원본 단위, 목표 단위를 받아 결정적 "
            "환산 결과를 반환하며, 요청 시 유효자리(precision)에 맞춰 반올림한다. 온도(섭씨/화씨/켈빈)는 "
            "선형 오프셋 변환을, 길이/무게/부피는 SI 기준 배율 변환을 사용한다. 레시피·엔지니어링·여행 등에서 "
            "빠른 단위 환산이 필요할 때 사용한다. 서로 다른 물리 차원(예: 미터 → 킬로그램)은 에러를 반환한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "변환할 수치. 예: 100",
                },
                "from_unit": {
                    "type": "string",
                    "description": "원본 단위.",
                    "enum": ["m", "km", "cm", "mm", "mi", "ft", "in", "kg", "g", "mg", "lb", "oz", "c", "f", "k", "l", "ml", "gal"],
                },
                "to_unit": {
                    "type": "string",
                    "description": "목표 단위. from_unit 과 같은 물리 차원이어야 함.",
                    "enum": ["m", "km", "cm", "mm", "mi", "ft", "in", "kg", "g", "mg", "lb", "oz", "c", "f", "k", "l", "ml", "gal"],
                },
                "precision": {
                    "type": "integer",
                    "description": "결과 소수점 자릿수(반올림). 기본값 4. 예: 2",
                },
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
    {
        "type": "function",
        "name": "generate_workout_routine",
        "description": (
            "목표 근육군과 운동 경력, 사용 가능한 장비를 기반으로 하루 운동 루틴(운동 종목·세트·반복수·휴식)을 "
            "생성한다. 초급/중급/고급에 따라 볼륨과 강도를 조절하고, 지정한 근육군을 우선 타깃하는 종목을 "
            "우선순위로 배치한다. 헬스장/홈트 등 장비 제약을 반영해 대체 종목을 고른다. 오늘 어떤 운동을 할지 "
            "빠르게 구성안을 받고 싶을 때 사용한다."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "muscle_group": {
                    "type": "string",
                    "description": "주 타깃 근육군.",
                    "enum": ["chest", "back", "legs", "shoulders", "arms", "core", "full_body"],
                },
                "experience_level": {
                    "type": "string",
                    "description": "운동 경력 수준. 볼륨/강도 조절 기준.",
                    "enum": ["beginner", "intermediate", "advanced"],
                },
                "equipment": {
                    "type": "string",
                    "description": "사용 가능한 장비 환경.",
                    "enum": ["full_gym", "dumbbells_only", "bodyweight", "resistance_band"],
                },
                "duration_min": {
                    "type": "integer",
                    "description": "목표 운동 시간(분). 종목 수 산정에 사용. 기본값 45. 예: 30",
                },
            },
            "required": ["muscle_group", "experience_level"],
        },
    },
]


def _suggest_ingredient_substitute(args: dict) -> str:
    ingredient = args.get("ingredient")
    if not ingredient:
        return "error: 'ingredient' is required"
    quantity = args.get("quantity", "unspecified")
    dish_type = args.get("dish_type", "general")
    dietary = args.get("dietary_restriction", "none")

    table = {
        "butter": [
            {"substitute": "olive_oil", "ratio": "3/4 of original", "note": "지방감은 유지되나 유제품 풍미는 약해짐"},
            {"substitute": "coconut_oil", "ratio": "1:1", "note": "베이킹 시 고형 상태로 계량, 은은한 코코넛 향"},
            {"substitute": "greek_yogurt", "ratio": "1/2 of original", "note": "수분이 늘어 굽는 시간 소폭 증가"},
        ],
        "buttermilk": [
            {"substitute": "milk_plus_lemon", "ratio": "1 cup milk + 1 tbsp lemon juice", "note": "10분 정도 두어 응고시킨 뒤 사용"},
            {"substitute": "plain_yogurt_thinned", "ratio": "1:1 (물로 희석)", "note": "산도 유지, 팬케이크에 적합"},
        ],
        "egg": [
            {"substitute": "flaxseed_meal", "ratio": "1 tbsp meal + 3 tbsp water = 1 egg", "note": "5분 불려 젤화, 결착용에 적합"},
            {"substitute": "mashed_banana", "ratio": "1/4 cup = 1 egg", "note": "단맛/향 추가, 팬케이크·머핀 한정"},
            {"substitute": "applesauce", "ratio": "1/4 cup = 1 egg", "note": "수분 결착, 부풀림은 약함"},
        ],
    }

    key = str(ingredient).strip().lower()
    aliases = {"버터": "butter", "계란": "egg", "달걀": "egg", "eggs": "egg", "버터밀크": "buttermilk"}
    key = aliases.get(key, key)

    candidates = table.get(key)
    if not candidates:
        candidates = [
            {"substitute": key + "_alt_generic", "ratio": "1:1", "note": "동일 계열 재료로 등량 대체 권장 (일반 규칙)"},
        ]

    # 식이 제약 필터 (결정적)
    banned = {
        "vegan": {"greek_yogurt", "plain_yogurt_thinned", "milk_plus_lemon", "egg"},
        "dairy_free": {"greek_yogurt", "plain_yogurt_thinned", "milk_plus_lemon", "buttermilk"},
        "gluten_free": set(),
        "nut_free": {"coconut_oil"},
        "none": set(),
    }
    block = banned.get(dietary, set())
    filtered = [c for c in candidates if c["substitute"] not in block] or candidates

    result = {
        "original_ingredient": ingredient,
        "quantity": quantity,
        "dish_type": dish_type,
        "dietary_restriction": dietary,
        "substitutes": filtered[:3],
    }
    return json.dumps(result, ensure_ascii=False)


def _git_blame_line(args: dict) -> str:
    repo = args.get("repo")
    file_path = args.get("file_path")
    line_start = args.get("line_start")
    if not repo or not file_path or line_start is None:
        return "error: 'repo', 'file_path', and 'line_start' are required"
    line_end = args.get("line_end", line_start)
    ref = args.get("ref", "HEAD")

    authors = ["dana.kim", "j.rivera", "wei.chen", "s.novak", "a.patel"]
    names = {
        "dana.kim": "Dana Kim",
        "j.rivera": "Jorge Rivera",
        "wei.chen": "Wei Chen",
        "s.novak": "Stefan Novak",
        "a.patel": "Anika Patel",
    }
    messages = [
        "fix: guard against null session token",
        "refactor: extract validation helper",
        "feat: add retry with backoff",
        "chore: bump dependency and reformat",
        "perf: cache compiled regex",
    ]

    seed = (len(repo) + len(file_path) + int(line_start)) % 5
    blame_lines = []
    for ln in range(int(line_start), int(line_end) + 1):
        pick = (seed + ln) % 5
        author = authors[pick]
        commit_hash = "%08x" % (((len(file_path) * 2654435761) ^ (ln * 40503) ^ (len(repo) * 97)) & 0xFFFFFFFF)
        day = (pick * 5 + ln) % 27 + 1
        month = (pick + 3) % 12 + 1
        blame_lines.append({
            "line": ln,
            "commit": commit_hash[:8],
            "author": author,
            "author_name": names[author],
            "authored_at": "2024-%02d-%02dT%02d:%02d:00Z" % (month, day, (ln * 3) % 24, (ln * 7) % 60),
            "summary": messages[pick],
            "snippet": "    return _validate(payload, strict=True)  # line %d" % ln,
        })

    result = {
        "repo": repo,
        "file_path": file_path,
        "ref": ref,
        "range": {"start": int(line_start), "end": int(line_end)},
        "blame": blame_lines,
    }
    return json.dumps(result, ensure_ascii=False)


def _format_legal_citation(args: dict) -> str:
    case_name = args.get("case_name")
    reporter = args.get("reporter")
    volume = args.get("volume")
    page = args.get("page")
    year = args.get("year")
    if not all([case_name, reporter, volume, page, year]):
        return "error: 'case_name', 'reporter', 'volume', 'page', and 'year' are all required"
    pincite = args.get("pincite")
    style = args.get("style", "bluebook")

    page_part = str(page)
    if pincite:
        page_part = "%s, %s" % (page, pincite)

    if style == "bluebook":
        citation = "%s, %s %s %s (%s)" % (case_name, volume, reporter, page_part, year)
    elif style == "korean_supreme":
        citation = "%s 판결 [%s %s면, %s]" % (case_name, reporter, page_part, year)
    elif style == "apa":
        citation = "%s, %s %s %s (%s)." % (case_name, volume, reporter, page_part, year)
    else:
        citation = "%s, %s %s %s (%s)" % (case_name, volume, reporter, page_part, year)

    result = {
        "style": style,
        "citation": citation,
        "components": {
            "case_name": case_name,
            "reporter": reporter,
            "volume": volume,
            "page": page,
            "pincite": pincite or None,
            "year": year,
        },
    }
    return json.dumps(result, ensure_ascii=False)


def _convert_unit(args: dict) -> str:
    value = args.get("value")
    from_unit = args.get("from_unit")
    to_unit = args.get("to_unit")
    if value is None or not from_unit or not to_unit:
        return "error: 'value', 'from_unit', and 'to_unit' are required"
    precision = args.get("precision", 4)
    try:
        value = float(value)
        precision = int(precision)
    except (TypeError, ValueError):
        return "error: 'value' must be numeric and 'precision' an integer"

    # SI 기준 배율 (같은 차원끼리만 변환 가능)
    length = {"m": 1.0, "km": 1000.0, "cm": 0.01, "mm": 0.001, "mi": 1609.344, "ft": 0.3048, "in": 0.0254}
    mass = {"kg": 1.0, "g": 0.001, "mg": 1e-6, "lb": 0.45359237, "oz": 0.028349523125}
    volume = {"l": 1.0, "ml": 0.001, "gal": 3.785411784}
    temp = {"c", "f", "k"}

    def dim(u):
        if u in length:
            return "length"
        if u in mass:
            return "mass"
        if u in volume:
            return "volume"
        if u in temp:
            return "temperature"
        return None

    d_from, d_to = dim(from_unit), dim(to_unit)
    if d_from is None or d_to is None:
        return "error: unknown unit"
    if d_from != d_to:
        return "error: incompatible dimensions (%s -> %s)" % (d_from, d_to)

    if d_from == "temperature":
        # 섭씨 기준으로 정규화
        if from_unit == "c":
            celsius = value
        elif from_unit == "f":
            celsius = (value - 32.0) * 5.0 / 9.0
        else:  # k
            celsius = value - 273.15
        if to_unit == "c":
            out = celsius
        elif to_unit == "f":
            out = celsius * 9.0 / 5.0 + 32.0
        else:  # k
            out = celsius + 273.15
    else:
        scale = {"length": length, "mass": mass, "volume": volume}[d_from]
        out = value * scale[from_unit] / scale[to_unit]

    out = round(out, precision)
    result = {
        "input": {"value": value, "unit": from_unit},
        "output": {"value": out, "unit": to_unit},
        "dimension": d_from,
        "precision": precision,
    }
    return json.dumps(result, ensure_ascii=False)


def _generate_workout_routine(args: dict) -> str:
    muscle_group = args.get("muscle_group")
    experience = args.get("experience_level")
    if not muscle_group or not experience:
        return "error: 'muscle_group' and 'experience_level' are required"
    equipment = args.get("equipment", "full_gym")
    duration = args.get("duration_min", 45)
    try:
        duration = int(duration)
    except (TypeError, ValueError):
        duration = 45

    exercises = {
        "chest": {
            "full_gym": ["barbell bench press", "incline dumbbell press", "cable fly", "chest dip"],
            "dumbbells_only": ["dumbbell bench press", "incline dumbbell press", "dumbbell fly", "floor press"],
            "bodyweight": ["push-up", "wide push-up", "decline push-up", "diamond push-up"],
            "resistance_band": ["band chest press", "band fly", "band incline press", "band push-up"],
        },
        "back": {
            "full_gym": ["deadlift", "lat pulldown", "seated cable row", "barbell row"],
            "dumbbells_only": ["dumbbell row", "renegade row", "dumbbell pullover", "reverse fly"],
            "bodyweight": ["pull-up", "inverted row", "superman", "prone Y-raise"],
            "resistance_band": ["band row", "band lat pulldown", "band pull-apart", "band deadlift"],
        },
        "legs": {
            "full_gym": ["back squat", "leg press", "romanian deadlift", "leg curl"],
            "dumbbells_only": ["goblet squat", "dumbbell lunge", "dumbbell RDL", "calf raise"],
            "bodyweight": ["air squat", "walking lunge", "glute bridge", "wall sit"],
            "resistance_band": ["band squat", "band lunge", "band leg curl", "band good morning"],
        },
        "shoulders": {
            "full_gym": ["overhead press", "lateral raise", "rear delt fly", "upright row"],
            "dumbbells_only": ["dumbbell shoulder press", "lateral raise", "front raise", "rear delt fly"],
            "bodyweight": ["pike push-up", "wall handstand hold", "arm circles", "scapular push-up"],
            "resistance_band": ["band overhead press", "band lateral raise", "band face pull", "band front raise"],
        },
        "arms": {
            "full_gym": ["barbell curl", "cable pushdown", "preacher curl", "skull crusher"],
            "dumbbells_only": ["dumbbell curl", "hammer curl", "overhead extension", "concentration curl"],
            "bodyweight": ["chin-up", "bench dip", "close push-up", "isometric curl"],
            "resistance_band": ["band curl", "band pushdown", "band hammer curl", "band overhead extension"],
        },
        "core": {
            "full_gym": ["cable crunch", "hanging leg raise", "ab wheel", "russian twist"],
            "dumbbells_only": ["weighted crunch", "dumbbell side bend", "russian twist", "toe touch"],
            "bodyweight": ["plank", "bicycle crunch", "leg raise", "mountain climber"],
            "resistance_band": ["band pallof press", "band woodchop", "band dead bug", "band crunch"],
        },
        "full_body": {
            "full_gym": ["barbell squat", "bench press", "barbell row", "overhead press"],
            "dumbbells_only": ["dumbbell thruster", "renegade row", "goblet squat", "dumbbell swing"],
            "bodyweight": ["burpee", "push-up", "air squat", "plank"],
            "resistance_band": ["band squat to press", "band row", "band deadlift", "band press"],
        },
    }

    params = {
        "beginner": {"sets": 3, "reps": "10-12", "rest_sec": 90},
        "intermediate": {"sets": 4, "reps": "8-10", "rest_sec": 75},
        "advanced": {"sets": 5, "reps": "5-8", "rest_sec": 60},
    }

    group_map = exercises.get(muscle_group)
    if group_map is None:
        return "error: unknown muscle_group '%s'" % muscle_group
    picks = group_map.get(equipment, group_map["full_gym"])
    cfg = params.get(experience, params["beginner"])

    # 시간에 따라 종목 수 산정 (결정적): 대략 종목당 10분
    n = max(2, min(len(picks), duration // 10))
    routine = []
    for ex in picks[:n]:
        routine.append({
            "exercise": ex,
            "sets": cfg["sets"],
            "reps": cfg["reps"],
            "rest_sec": cfg["rest_sec"],
        })

    result = {
        "muscle_group": muscle_group,
        "experience_level": experience,
        "equipment": equipment,
        "duration_min": duration,
        "estimated_exercises": n,
        "routine": routine,
    }
    return json.dumps(result, ensure_ascii=False)


IMPL = {
    "suggest_ingredient_substitute": _suggest_ingredient_substitute,
    "git_blame_line": _git_blame_line,
    "format_legal_citation": _format_legal_citation,
    "convert_unit": _convert_unit,
    "generate_workout_routine": _generate_workout_routine,
}

# ── 그룹 D3 캡처 ──
TOOLS_D3, IMPL_D3 = TOOLS, IMPL
print("D3(방해) 도구:", [t["name"] for t in TOOLS_D3])


## 네임스페이스 병합 + 디스패처

In [ ]:
TOOLS_FULL = TOOLS_REL + TOOLS_D1 + TOOLS_D2 + TOOLS_D3
TOOLS_CLEAN = TOOLS_REL
IMPL = {}
for _d in (IMPL_REL, IMPL_D1, IMPL_D2, IMPL_D3):
    IMPL.update(_d)
RELEVANT_NAMES = {t["name"] for t in TOOLS_REL}
assert len(TOOLS_FULL) == 20 and len(IMPL) == 20, (len(TOOLS_FULL), len(IMPL))

def run_tool(name, args):
    fn = IMPL.get(name)
    if fn is None:
        return f"<error>알 수 없는 도구: {name}</error>"
    try:
        return fn(args)
    except Exception as e:
        return f"<error>{type(e).__name__}: {e}</error>"

print(f"네임스페이스: 전체 {len(TOOLS_FULL)}개 (관련 {len(TOOLS_REL)} + 방해 {len(TOOLS_FULL)-len(TOOLS_REL)}) · 클린 {len(TOOLS_CLEAN)}개")

## 평가 과제 (Stage 1) — 관련 5개 도구를 조합해야 풀린다

방해 15개는 어느 과제에도 필요 없다. 검증기는 정답에 반드시 들어갈 문자열(금액·charge_id·환불id)이
모두 있는지만 본다(느슨, 콤마·대소문자 무시).

In [ ]:
@dataclass
class Task:
    id: str
    steps: int
    prompt: str
    must_include: list
    must_exclude: list = field(default_factory=list)   # 있으면 실패 (예: 과환불)

    def verify(self, answer):
        low = (answer or "").lower().replace(",", "")
        missing = [s for s in self.must_include if s.lower().replace(",", "") not in low]
        wrong = [s for s in self.must_exclude if s.lower().replace(",", "") in low]
        if missing:
            return False, "누락: " + ", ".join(missing)
        if wrong:
            return False, "있으면 안 되는 것 포함(예: 정상청구 과환불): " + ", ".join(wrong)
        return True, "정답 요소 충족"

EVAL_TASKS = [
    Task("N1", 2, "고객 sarah@example.com 의 'paid' 상태 주문들의 총액 합(원)을 구하라.", ["57000"]),
    Task("N2", 3, "고객 sarah@example.com 의 주문 ORD-1002 에 중복 청구(같은 금액 succeeded 가 2건 이상)가 있는지 확인하고, 가장 먼저 성공한 1건은 정상으로 두고 나머지 중복 charge_id 를 모두 답하라.", ["CHG-3", "CHG-4"]),
    # N3: 정상 1건(CHG-2)은 남기고 중복만 환불해야 함 → 과환불(RFND-CHG-2) 은 must_exclude 로 실패 처리
    Task("N3", 5, "고객 sarah@example.com 이 주문 ORD-1002 에 3번(CHG-2·CHG-3·CHG-4) 청구됐다. 정상 1건인 가장 먼저 성공한 CHG-2 는 그대로 두고, 중복분만 환불한 뒤 고객에게 이메일로 알리고, 환불 확인 id 들과 알림 id 를 답하라.",
         ["RFND-CHG-3", "RFND-CHG-4"], ["RFND-CHG-2"]),
    Task("N4", 2, "고객 minjun@example.com 의 취소된(cancelled) 주문의 금액(원)을 알려줘.", ["30000"]),
]
print("과제", len(EVAL_TASKS), "개 (N3 은 과환불 검출 엄격 검증)")

## 실행 하네스 — 오호출(방해도구 호출) 카운트 (Stage 2)

관련 없는 도구를 부르면 wrong_tool_calls 로 집계한다. 입력 토큰(매 턴 스키마 비용)도 기록.

In [ ]:
SYSTEM_PROMPT = (
    "너는 전자상거래 고객지원 운영 에이전트다. 제공된 도구만으로 사용자 요청을 처리하라. "
    "관련 없는 도구는 호출하지 말고, 필요한 정보는 반드시 도구로 조회한 뒤 결론을 내려라(추측 금지).\n"
    "1) 도구 호출 앞에 <plan>...</plan> 으로 다음 스텝을 한 줄 적어라.\n"
    "2) 마지막에 <answer>...</answer> 로 최종 답을 명확히 적어라(요청된 id·금액 등 포함).\n"
)

def run_task(client, task, tools, arm="", model=None, max_turns=15):
    model = model or MODEL
    input_list = [{"role": "user", "content": task.prompt}]
    called = []
    n_calls = n_err = in_tok = out_tok = turns = 0
    final = ""
    t0 = time.time()
    for _ in range(max_turns):
        resp = client.responses.create(model=model, instructions=SYSTEM_PROMPT,
                                       input=input_list, tools=tools, parallel_tool_calls=True)
        turns += 1
        if resp.usage:
            in_tok += resp.usage.input_tokens
            out_tok += resp.usage.output_tokens
        input_list += resp.output
        calls = [it for it in resp.output if it.type == "function_call"]
        if not calls:
            final = resp.output_text
            break
        for c in calls:
            try:
                args = json.loads(c.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            res = run_tool(c.name, args)
            n_calls += 1
            called.append(c.name)
            if res.startswith("<error>"):
                n_err += 1
            input_list.append({"type": "function_call_output", "call_id": c.call_id, "output": res})
    else:
        final = final or "(max_turns 도달)"
    passed, reason = task.verify(final)
    wrong = [n for n in called if n not in RELEVANT_NAMES]
    return {"arm": arm, "task_id": task.id, "passed": passed, "reason": reason, "turns": turns,
            "tool_calls": n_calls, "tool_errors": n_err, "wrong_tool_calls": len(wrong),
            "wrong_names": sorted(set(wrong)), "input_tokens": in_tok, "output_tokens": out_tok,
            "duration_s": round(time.time() - t0, 2), "called": called, "final_text": final}

## A/B 실행 — full(20) vs clean(5) (Stage 2·3)

같은 과제를 두 네임스페이스에 돌린다. 방해 15개가 정확도·오호출·입력토큰에 주는 영향을 본다.

In [ ]:
ARMS = [("full(20: 관련5+방해15)", TOOLS_FULL), ("clean(관련5만)", TOOLS_CLEAN)]
TRIALS = 1

if client:
    allr = []
    for label, tools in ARMS:
        rs = [run_task(client, t, tools, label) for t in EVAL_TASKS for _ in range(TRIALS)]
        allr += rs
        n = len(rs)
        print(f"{label:22} 정확도 {sum(r['passed'] for r in rs)/n:4.0%} · 오호출(방해) {sum(r['wrong_tool_calls'] for r in rs)} · "
              f"평균 호출 {sum(r['tool_calls'] for r in rs)/n:.1f} · turns {sum(r['turns'] for r in rs)/n:.1f} · "
              f"평균 입력토큰 {sum(r['input_tokens'] for r in rs)/n:,.0f} · 총토큰 {sum(r['input_tokens']+r['output_tokens'] for r in rs)/n:,.0f}")
    print("\n과제별 (full vs clean):")
    for t in EVAL_TASKS:
        fu = [r for r in allr if r['arm'] == ARMS[0][0] and r['task_id'] == t.id]
        cl = [r for r in allr if r['arm'] == ARMS[1][0] and r['task_id'] == t.id]
        wn = sorted({n for r in fu for n in r['wrong_names']})
        print(f"  {t.id}(steps {t.steps}) | full {sum(r['passed'] for r in fu)}/{len(fu)} 오호출{sum(r['wrong_tool_calls'] for r in fu)}{(' '+str(wn)) if wn else ''} | clean {sum(r['passed'] for r in cl)}/{len(cl)}")
    outdir = pathlib.Path.cwd() / "results"; outdir.mkdir(exist_ok=True)
    out = outdir / f"namespace-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    out.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in allr))
    print(f"\nTRIALS={TRIALS} · 저장:", out)
else:
    print("OPENAI_API_KEY 없음 — 이 셀 건너뜀")

## 구조 분석 (API 불필요)

In [ ]:
print("전체 네임스페이스:", len(TOOLS_FULL), "개")
print("  관련(relevant):", sorted(RELEVANT_NAMES))
print("  방해(distractor):", [t["name"] for t in TOOLS_FULL if t["name"] not in RELEVANT_NAMES])
full_chars = len(json.dumps(TOOLS_FULL, ensure_ascii=False))
clean_chars = len(json.dumps(TOOLS_CLEAN, ensure_ascii=False))
print(f"\n도구 스키마 크기(문자): full {full_chars:,} vs clean {clean_chars:,} → 매 턴 프롬프트에 약 {full_chars/clean_chars:.1f}배")
print("\n관련 도구 결정적 확인(정답 데이터):")
print("  find_customer(sarah):", run_tool("find_customer", {"query": "sarah@example.com"}))
print("  get_charges(ORD-1002):", run_tool("get_charges", {"order_id": "ORD-1002"}))
print("  issue_refund(CHG-3):", run_tool("issue_refund", {"charge_id": "CHG-3", "reason": "중복"}))

## 결론 — 큰 혼합 네임스페이스가 툴콜링에 주는 영향

- **토큰(구조적 비용)**: 관련 없는 15개 도구도 **매 요청 프롬프트에 스키마가 전부 실린다.** full 은 clean 대비 스키마가 여러 배 크고, 멀티턴이면 매 턴 그만큼 더 든다(위 입력토큰·스키마 크기 비교).
- **오호출(정확도 리스크)**: 네임스페이스가 크고 헷갈리는 도구가 많을수록 **엉뚱한 도구를 부를 확률**이 커진다(wrong_tool_calls). 여기선 도메인이 확연히 달라 오호출이 적을 수 있지만, 이름이 비슷한 도구가 섞이면 급증한다 — 그게 진짜 위험 구간.
- **실측에서 정확도도 갈렸다(엄격 채점 시)**: 첫 실행에서 N3 를 'RFND-CHG-3·4 포함'만으로 채점하니 full 이 100% 로 보였다. 그러나 로그를 보니 full 은 정상 청구 CHG-2 까지 환불(과환불, `issue_refund` 3회)했고 느슨한 검증기가 이를 놓쳤다. `must_exclude`(RFND-CHG-2 있으면 실패)로 조이자 **full 3/4(75%) vs clean 4/4(100%)** — 즉 큰 네임스페이스가 까다로운 멀티턴에서 실제로 실행을 무너뜨렸다(n=1, 반복 확인 필요). 교훈: **집계 숫자만 믿지 말고 로그를 읽고 검증기를 조여라**(지나치게 느슨한 검증기 금지).
- **해법 = 네임스페이스를 좁혀라**: 클로드코드의 **ToolSearch / deferred tools** 가 바로 이 문제의 답이다 — 도구를 전부 상주시키지 않고, 필요할 때 검색해 그 순간 관련 도구만 올린다(프리픽스 캐시도 보존). 즉 하네스가 "20개 중 5개만 관련" 상황을 자동으로 clean(5) 로 만들어 준다.
- **원칙(Anthropic 가이드)**: 도구는 많다고 좋은 게 아니다. **한 작업에서 에이전트가 실제로 고려할 도구 수를 줄여라** — 관련 도구는 통합(consolidation)으로 합치고, 무관 도구는 지연 로딩(ToolSearch)으로 빼라. 이 노트북의 full↔clean 격차가 그 이득의 크기다.

> 방해 15개 도구는 하위 서브에이전트 3개가 **병렬**로 작성한 실제 도메인별 도구다. 관련 5개(고객지원 워크플로)와 완전히 다른 컨텍스트라 '진짜 혼합 네임스페이스'를 이룬다.